In [1]:
import requests
import numpy as np
import pandas as pd
import random
from bs4 import BeautifulSoup
import pybaseball
import warnings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
#Pull player information into df
initialPull = pybaseball.chadwick_register()

Gathering player lookup table. This may take a moment.


In [3]:
#Initial Data Cleaning
cols = ['name_first', 'name_last', 'key_bbref', 'mlb_played_first', 'mlb_played_last']
initialPull = initialPull[cols]
initialPull['numSeasons'] = initialPull['mlb_played_last'] - initialPull['mlb_played_first']
cleanRows = initialPull.dropna(subset = ['key_bbref'])

<h1><center>THE GAME BEGINS HERE</center></h1>

In [4]:
#Function to scrape all time team lists from bref
def pullAlltimePlayers(team):
    #Request all time team list
    url = f'https://www.baseball-reference.com/teams/{team}/bat.shtml'
    r = requests.get(url)
    data = r.text
    teamQuery = BeautifulSoup(data, 'html.parser')

    #Find table element
    table = teamQuery.find('tbody')

    teamPlayers = []

    #Iterate through all rows and retrieve the player id value to insert into our list
    for i in table.find_all('td', class_ = 'left', attrs= {'data-stat': 'player'}):
        teamPlayers.append(i['data-append-csv'])

    return teamPlayers

In [5]:
#Function that accepts filter values from user and generates random player accordingly
def generateNewPlayer(cleanRows, seasonValue=None, debutValue=None,retirementValue=None, teamValue = None):
    #Check if None vals
    if seasonValue is None:
        seasonValue = 0

    if debutValue is None:
        debutValue = 0

    if retirementValue is None:
        retirementValue = 10000

    #Input filters
    filteredDF = cleanRows[(cleanRows['numSeasons'] >= seasonValue) & (cleanRows['mlb_played_first'] >= debutValue) & (cleanRows['mlb_played_last'] <= retirementValue)]

    #Prototype of filtering by team
    if teamValue is not None:
        teamList = pullAlltimePlayers(teamValue)
        filteredDF = filteredDF[filteredDF['key_bbref'].isin(teamList)]

    #Get new random ID and get bref url
    chosenPlayerID = filteredDF['key_bbref'].sample(n=1).iloc[0]
    baseLink = f'https://www.baseball-reference.com/players/{chosenPlayerID[0]}/{chosenPlayerID}.shtml'
    
    return baseLink, chosenPlayerID

In [6]:
#Function that processes bref page and returns final df
def getNewPlayer(baseLink):
    #Scrape chosen player
    url = baseLink
    r = requests.get(url)
    data = r.text
    baseball = BeautifulSoup(data, 'html.parser')
    
    #Find player stats table
    table = baseball.find('table', class_ = 'stats_table sortable row_summable suppress_headers soc')
 
    #Cycle through table and find correct elements
    entriesTR = []

    for i in table.find_all('tr'):
        entriesTR.append(i.text)

    #Create list of lists for future data frame compilation
    playerList = []

    for i in range(len(entriesTR)):
        inList = []
        rows = entriesTR[i].split(' ')
        for j in rows:
            inList.append(j)

        playerList.append(inList)

    #Split lists into individual elements
    for i in range(len(entriesTR)):
        playerList[i] = entriesTR[i].split(' ')

    #Convert to DF and remove unnecessary cols/rows
    data = pd.DataFrame(playerList)
    data.columns = data.iloc[0].values
    cutoffRow = data.index[(data['Age'] == 'Yrs') | (data['Age'] == 'Yr') | (data['Age'] == 'WAR')]
    data = data.iloc[1:cutoffRow[0]]
    data = data.fillna(' ')
    data = data.drop(data[''], axis = 1)

    return data

In [9]:
#IT'S TIME TO PLAY THE GAME
baseLink, chosenPlayerID = generateNewPlayer(cleanRows=cleanRows, seasonValue=3, debutValue=2016, teamValue='SEA', retirementValue=2025)
getNewPlayer(baseLink)

,Season,Age,Team,Lg,WAR,W,L,W-L%,ERA,G,GS,GF,CG,SHO,SV,IP,H,R,ER,HR,BB,IBB,SO,HBP,BK,WP,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/BB,Awards
1,2018,21,LAD,NL,0.3,7,2,.778,3.49,29,3,7,0,0,2,49.0,43,21,19,8,12,1,59,3,0,1,202,112,3.79,1.122,7.9,1.5,2.2,10.8,4.92,
2,2019,22,LAD,NL,-0.2,1,2,.333,4.84,46,2,5,0,0,0,44.2,39,26,24,7,27,2,54,6,1,1,204,86,5.05,1.478,7.9,1.4,5.4,10.9,2.00,
3,2020,23,LAD,NL,0.3,2,1,.667,2.89,21,1,0,0,0,0,18.2,16,7,6,4,3,0,27,0,0,0,75,153,3.57,1.018,7.7,1.9,1.4,13.0,9.00,
4,2021,Did,not,play,-,Injury,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
5,2022,25,LAD,NL,0.7,1,0,1.000,1.82,37,1,3,0,0,0,34.2,23,9,7,1,17,0,37,2,0,0,142,223,3.00,1.154,6.0,0.3,4.4,9.6,2.18,
6,2023,26,LAD,NL,0.1,7,4,.636,3.43,68,7,9,0,0,3,60.1,64,30,23,4,23,0,70,8,0,3,270,126,3.34,1.442,9.5,0.6,3.4,10.4,3.04,
7,2024,27,2TM,AL,-0.5,1,4,.200,4.64,62,0,19,0,0,1,54.1,56,34,28,6,25,1,67,4,0,3,249,88,3.74,1.491,9.3,1.0,4.1,11.1,2.68,
8,2024,27,NYY,AL,-0.7,1,3,.250,5.13,42,0,12,0,0,1,33.1,34,24,19,5,16,1,41,2,0,2,155,79,4.28,1.500,9.2,1.4,4.3,11.1,2.56,
9,2024,27,HOU,AL,0.2,0,1,.000,3.86,20,0,7,0,0,0,21.0,22,10,9,1,9,0,26,2,0,1,94,106,2.88,1.476,9.4,0.4,3.9,11.1,2.89,
10,2025,28,2TM,2LG,0.9,5,4,.556,3.58,70,0,4,0,0,0,65.1,54,27,26,2,22,3,51,6,0,1,270,118,3.26,1.163,7.4,0.3,3.0,7.0,2.32,


In [10]:
#RETRIEVE MYSTERY PLAYER NAME AND BREF LINK
answer = cleanRows[cleanRows['key_bbref'] == chosenPlayerID].values
print(f'{answer[0][0]} {answer[0][1]}')


Caleb Ferguson
